In [1]:
from statsbombpy import sb
import pandas as pd
import numpy as np

In [33]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

print(project_root)

d:\Temperance_football_analysis_ai_agent


In [2]:
match_id = 3754296
events = sb.events(match_id=match_id)

d:\Temperance_football_analysis_ai_agent\.venv\Lib\site-packages\statsbombpy\api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


In [3]:
def get_team_shots(events, team):
    return events[
        (events["type"] == "Shot") &
        (events["team"] == team)
    ].copy()

In [4]:
arsenal_shots = get_team_shots(events, "Arsenal")

arsenal_shots[[
    "minute",
    "player",
    "location",
    "shot_outcome",
    "shot_statsbomb_xg"
]].head()

,minute,player,location,shot_outcome,shot_statsbomb_xg
3730,32,Theo Walcott,"[100.9, 25.3]",Goal,0.034853
3732,45,Olivier Giroud,"[111.7, 26.5]",Goal,0.103572
3735,52,Joel Nathaniel Campbell Samuels,"[111.8, 37.5]",Off T,0.214126
3737,55,Joel Nathaniel Campbell Samuels,"[112.6, 53.3]",Saved,0.071731
3738,59,Aaron Ramsey,"[111.4, 36.2]",Saved,0.495885


In [5]:
def summarize_shots(events):
    shots = events[events["type"] == "Shot"].copy()

    summary = (
        shots.groupby("team")["shot_statsbomb_xg"]
        .agg(
            shots="count",
            total_xg="sum",
            avg_xg="mean"
        )
    )

    return summary

In [6]:
summarize_shots(events)

,shots,total_xg,avg_xg
team,,,
Arsenal,8,1.353976,0.169247
Manchester City,19,1.249865,0.065782


In [7]:
def get_team_passes(events, team):
    return events[
        (events["type"] == "Pass") &
        (events["team"] == team)
    ].copy()

In [8]:
passes = get_team_passes(events, "Arsenal")

[col for col in passes.columns if "pass" in col]

['pass_aerial_won',
 'pass_angle',
 'pass_assisted_shot_id',
 'pass_body_part',
 'pass_cross',
 'pass_cut_back',
 'pass_end_location',
 'pass_goal_assist',
 'pass_height',
 'pass_inswinging',
 'pass_length',
 'pass_no_touch',
 'pass_outcome',
 'pass_outswinging',
 'pass_recipient',
 'pass_recipient_id',
 'pass_shot_assist',
 'pass_switch',
 'pass_technique',
 'pass_through_ball',
 'pass_type',
 'shot_key_pass_id']

In [10]:
def summarize_passes(events):
    passes = events[events["type"] == "Pass"].copy()

    passes["completed"] = passes["pass_outcome"].isna()

    summary = (
        passes.groupby("team")
        .agg(
            passes=("id", "count"),
            completed_passes=("completed", "sum")
        )
    )

    summary["completion_rate"] = (
        summary["completed_passes"] / summary["passes"] * 100
    )

    return summary

In [11]:
summarize_passes(events)

,passes,completed_passes,completion_rate
team,,,
Arsenal,427,319,74.707260
Manchester City,705,593,84.113475


In [12]:
arsenal_passes = get_team_passes(events, "Arsenal")

arsenal_passes[[
    "minute",
    "player",
    "pass_recipient",
    "pass_length",
    "pass_outcome"
]].head(20)

,minute,player,pass_recipient,pass_length,pass_outcome
6,0,Olivier Giroud,Mesut Özil,0.860233,NaN
7,0,Mesut Özil,Laurent Koscielny,14.905368,NaN
8,0,Laurent Koscielny,Theo Walcott,55.506847,Incomplete
10,0,Ignacio Monreal Eraso,Olivier Giroud,25.278646,NaN
11,0,Olivier Giroud,NaN,2.300000,Incomplete
12,0,Ignacio Monreal Eraso,Mesut Özil,4.119466,NaN
13,0,Mesut Özil,Ignacio Monreal Eraso,2.607681,NaN
14,0,Ignacio Monreal Eraso,Mesut Özil,4.687217,NaN
15,0,Mesut Özil,Aaron Ramsey,8.415462,NaN
16,0,Aaron Ramsey,NaN,7.623647,Out


In [13]:
arsenal_passes[
    arsenal_passes["pass_outcome"].notna()
][[
    "minute",
    "player",
    "pass_recipient",
    "pass_length",
    "pass_outcome"
]].head(20)

,minute,player,pass_recipient,pass_length,pass_outcome
8,0,Laurent Koscielny,Theo Walcott,55.506847,Incomplete
11,0,Olivier Giroud,NaN,2.300000,Incomplete
16,0,Aaron Ramsey,NaN,7.623647,Out
18,1,Ignacio Monreal Eraso,NaN,20.664946,Incomplete
21,1,Ignacio Monreal Eraso,NaN,15.717824,Incomplete
24,1,Joel Nathaniel Campbell Samuels,Olivier Giroud,16.788687,Pass Offside
28,2,Mathieu Flamini,NaN,14.400000,Incomplete
61,4,Laurent Koscielny,Olivier Giroud,26.271849,Incomplete
99,7,Olivier Giroud,Joel Nathaniel Campbell Samuels,23.220680,Incomplete
101,7,Per Mertesacker,NaN,23.685650,Incomplete


In [14]:
def get_final_third_entries(events, team):
    passes = events[
        (events["type"] == "Pass") &
        (events["team"] == team)
    ].copy()

    passes = passes[
        passes["location"].notna() &
        passes["pass_end_location"].notna()
    ].copy()

    passes["start_x"] = passes["location"].apply(lambda x: x[0])
    passes["end_x"] = passes["pass_end_location"].apply(lambda x: x[0])

    entries = passes[
        (passes["start_x"] < 80) &
        (passes["end_x"] >= 80)
    ].copy()

    return entries

In [15]:
arsenal_entries = get_final_third_entries(events, "Arsenal")
city_entries = get_final_third_entries(events, "Manchester City")

print("Arsenal final-third entries:", len(arsenal_entries))
print("Manchester City final-third entries:", len(city_entries))

Arsenal final-third entries: 52
Manchester City final-third entries: 60


In [16]:
arsenal_entries[[
    "minute",
    "player",
    "pass_recipient",
    "location",
    "pass_end_location"
]].head(20)

,minute,player,pass_recipient,location,pass_end_location
8,0,Laurent Koscielny,Theo Walcott,"[42.6, 39.5]","[85.6, 4.4]"
18,1,Ignacio Monreal Eraso,NaN,"[70.1, 4.6]","[85.3, 18.6]"
21,1,Ignacio Monreal Eraso,NaN,"[70.7, 5.7]","[86.0, 9.3]"
34,2,Héctor Bellerín Moruno,Mesut Özil,"[76.6, 66.5]","[91.0, 54.7]"
49,3,Aaron Ramsey,Olivier Giroud,"[63.1, 70.7]","[106.8, 56.8]"
99,7,Olivier Giroud,Joel Nathaniel Campbell Samuels,"[69.6, 69.4]","[92.4, 73.8]"
113,7,Laurent Koscielny,Mesut Özil,"[70.4, 17.8]","[82.7, 17.5]"
157,10,Laurent Koscielny,Theo Walcott,"[70.1, 33.8]","[109.9, 0.1]"
160,11,Aaron Ramsey,Mesut Özil,"[78.0, 8.1]","[84.0, 10.4]"
184,13,Petr Čech,Olivier Giroud,"[35.8, 12.2]","[97.4, 17.2]"


In [17]:
def summarize_final_third_entries(events):
    teams = events["team"].dropna().unique()

    results = []

    for team in teams:
        entries = get_final_third_entries(events, team)

        results.append({
            "team": team,
            "final_third_entries": len(entries)
        })

    return pd.DataFrame(results)

In [18]:
def summarize_final_third_entries(events):
    teams = events["team"].dropna().unique()

    results = []

    for team in teams:
        entries = get_final_third_entries(events, team)

        results.append({
            "team": team,
            "final_third_entries": len(entries)
        })

    return pd.DataFrame(results)

In [19]:
summarize_final_third_entries(events)

,team,final_third_entries
0,Arsenal,52
1,Manchester City,60


In [20]:
def build_team_summary(events, team):
    shots = get_team_shots(events, team)
    passes = get_team_passes(events, team)
    entries = get_final_third_entries(events, team)

    completed_passes = passes["pass_outcome"].isna().sum()

    return {
        "team": team,
        "shots": len(shots),
        "total_xg": round(shots["shot_statsbomb_xg"].sum(), 3),
        "average_xg_per_shot": round(shots["shot_statsbomb_xg"].mean(), 3),
        "passes": len(passes),
        "completed_passes": int(completed_passes),
        "pass_completion_rate": round(
            completed_passes / len(passes) * 100, 2
        ),
        "final_third_entries": len(entries)
    }

In [22]:
arsenal_summary = build_team_summary(events, "Arsenal")
city_summary = build_team_summary(events, "Manchester City")

arsenal_summary
city_summary

{'team': 'Manchester City',
 'shots': 19,
 'total_xg': np.float64(1.25),
 'average_xg_per_shot': np.float64(0.066),
 'passes': 705,
 'completed_passes': 593,
 'pass_completion_rate': np.float64(84.11),
 'final_third_entries': 60}

In [23]:
def compare_teams(events, team_a, team_b):
    return {
        team_a: build_team_summary(events, team_a),
        team_b: build_team_summary(events, team_b)
    }

In [24]:
match_summary = compare_teams(
    events,
    "Arsenal",
    "Manchester City"
)

match_summary

{'Arsenal': {'team': 'Arsenal',
  'shots': 8,
  'total_xg': np.float64(1.354),
  'average_xg_per_shot': np.float64(0.169),
  'passes': 427,
  'completed_passes': 319,
  'pass_completion_rate': np.float64(74.71),
  'final_third_entries': 52},
 'Manchester City': {'team': 'Manchester City',
  'shots': 19,
  'total_xg': np.float64(1.25),
  'average_xg_per_shot': np.float64(0.066),
  'passes': 705,
  'completed_passes': 593,
  'pass_completion_rate': np.float64(84.11),
  'final_third_entries': 60}}

In [25]:
import math

def get_progressive_passes(events, team, min_progress=10):
    passes = events[
        (events["type"] == "Pass") &
        (events["team"] == team) &
        (events["pass_outcome"].isna())
    ].copy()

    passes = passes[
        passes["location"].notna() &
        passes["pass_end_location"].notna()
    ].copy()

    def distance_to_goal(location):
        x, y = location
        goal_x, goal_y = 120, 40
        return math.sqrt((goal_x - x)**2 + (goal_y - y)**2)

    passes["start_goal_distance"] = passes["location"].apply(distance_to_goal)
    passes["end_goal_distance"] = passes["pass_end_location"].apply(distance_to_goal)

    passes["progress"] = (
        passes["start_goal_distance"] -
        passes["end_goal_distance"]
    )

    progressive = passes[
        passes["progress"] >= min_progress
    ].copy()

    return progressive

In [26]:
arsenal_progressive = get_progressive_passes(events, "Arsenal")
city_progressive = get_progressive_passes(events, "Manchester City")

print("Arsenal progressive passes:", len(arsenal_progressive))
print("Manchester City progressive passes:", len(city_progressive))

Arsenal progressive passes: 85
Manchester City progressive passes: 123


In [27]:
arsenal_progressive[[
    "minute",
    "player",
    "pass_recipient",
    "location",
    "pass_end_location",
    "progress"
]].sort_values("progress", ascending=False).head(15)

,minute,player,pass_recipient,location,pass_end_location,progress
985,79,Petr Čech,Olivier Giroud,"[14.5, 39.8]","[78.6, 31.2]",63.175256
788,63,Petr Čech,Theo Walcott,"[5.6, 36.8]","[68.6, 42.5]",62.983985
550,42,Petr Čech,Olivier Giroud,"[6.0, 36.0]","[73.6, 18.3]",62.846624
1065,86,Petr Čech,Olivier Giroud,"[7.0, 44.1]","[70.5, 19.0]",59.304011
1080,88,Petr Čech,Olivier Giroud,"[7.0, 44.1]","[70.2, 64.8]",57.440913
184,13,Petr Čech,Olivier Giroud,"[35.8, 12.2]","[97.4, 17.2]",56.567667
1037,83,Petr Čech,Olivier Giroud,"[9.6, 35.2]","[76.2, 7.5]",55.963580
98,7,Petr Čech,Olivier Giroud,"[6.0, 44.0]","[69.6, 69.4]",55.721889
715,55,Laurent Koscielny,Joel Nathaniel Campbell Samuels,"[39.5, 47.1]","[105.3, 60.5]",55.586719
1100,90,Petr Čech,Alex Oxlade-Chamberlain,"[7.0, 36.1]","[71.3, 73.7]",53.844143


In [28]:
arsenal_progressive[[
    "minute",
    "player",
    "pass_recipient",
    "location",
    "pass_end_location",
    "progress"
]].sort_values("progress", ascending=False).head(15)

,minute,player,pass_recipient,location,pass_end_location,progress
985,79,Petr Čech,Olivier Giroud,"[14.5, 39.8]","[78.6, 31.2]",63.175256
788,63,Petr Čech,Theo Walcott,"[5.6, 36.8]","[68.6, 42.5]",62.983985
550,42,Petr Čech,Olivier Giroud,"[6.0, 36.0]","[73.6, 18.3]",62.846624
1065,86,Petr Čech,Olivier Giroud,"[7.0, 44.1]","[70.5, 19.0]",59.304011
1080,88,Petr Čech,Olivier Giroud,"[7.0, 44.1]","[70.2, 64.8]",57.440913
184,13,Petr Čech,Olivier Giroud,"[35.8, 12.2]","[97.4, 17.2]",56.567667
1037,83,Petr Čech,Olivier Giroud,"[9.6, 35.2]","[76.2, 7.5]",55.963580
98,7,Petr Čech,Olivier Giroud,"[6.0, 44.0]","[69.6, 69.4]",55.721889
715,55,Laurent Koscielny,Joel Nathaniel Campbell Samuels,"[39.5, 47.1]","[105.3, 60.5]",55.586719
1100,90,Petr Čech,Alex Oxlade-Chamberlain,"[7.0, 36.1]","[71.3, 73.7]",53.844143


In [29]:
def build_team_summary(events, team):
    shots = get_team_shots(events, team)
    passes = get_team_passes(events, team)
    entries = get_final_third_entries(events, team)
    progressive = get_progressive_passes(events, team)

    completed_passes = passes["pass_outcome"].isna().sum()

    return {
        "team": team,
        "shots": len(shots),
        "total_xg": round(shots["shot_statsbomb_xg"].sum(), 3),
        "average_xg_per_shot": round(shots["shot_statsbomb_xg"].mean(), 3),
        "passes": len(passes),
        "completed_passes": int(completed_passes),
        "pass_completion_rate": round(
            completed_passes / len(passes) * 100, 2
        ),
        "final_third_entries": len(entries),
        "progressive_passes": len(progressive)
    }

In [30]:
match_summary = compare_teams(
    events,
    "Arsenal",
    "Manchester City"
)

match_summary

{'Arsenal': {'team': 'Arsenal',
  'shots': 8,
  'total_xg': np.float64(1.354),
  'average_xg_per_shot': np.float64(0.169),
  'passes': 427,
  'completed_passes': 319,
  'pass_completion_rate': np.float64(74.71),
  'final_third_entries': 52,
  'progressive_passes': 85},
 'Manchester City': {'team': 'Manchester City',
  'shots': 19,
  'total_xg': np.float64(1.25),
  'average_xg_per_shot': np.float64(0.066),
  'passes': 705,
  'completed_passes': 593,
  'pass_completion_rate': np.float64(84.11),
  'final_third_entries': 60,
  'progressive_passes': 123}}

In [32]:
import os
print(os.getcwd())

d:\Temperance_football_analysis_ai_agent\notebooks


In [38]:
from src.analysis_tools import build_team_summary
from src.tool_registry import TOOL_DESCRIPTIONS

print(TOOL_DESCRIPTIONS.keys())

dict_keys(['team_summary', 'shot_analysis', 'final_third_entries', 'progressive_passes'])
